# exp_021 · Level 3 — Transformer Hidden States (`hidden_states`)

The deepest analysis level: internal representations inside LTX-2's DiT transformer blocks.

---

### What is a transformer block in LTX-2?

LTX-2 is a **Diffusion Transformer (DiT)** with **48 transformer blocks** (attention + FFN).
We probe 4 depths:

| Label | Block index (of 48) | Depth | Expected role |
|-------|---------------------|-------|---------------|
| **L12** | Block 12 | 25% | Shallow — low-level spatial/positional patterns |
| **L24** | Block 24 | 50% | Mid — spatial + semantic mixing |
| **L35** | Block 35 | 73% | Deep — high-level semantic content |
| **L47** | Block 47 | 98% | Near-output — direct precursor to velocity prediction |

---

### What is a token in this context?

LTX-2 packs all spatial positions of all latent frames into one flat 1D token sequence:

```
Sequence length = F' × H' × W' = 16 × 16 × 24 = 6,144 tokens
```

Token `(p × 16 × 24) + (h × 24) + w` corresponds to latent frame `p`, height `h`, width `w`.

---

### Saved hidden states

```
hidden_states[τ][L].shape = [2, 6144, 4096]
                              ↑   ↑      ↑
                         CFG  tokens  hidden dim
```

- `[0]` = unconditional forward pass (CFG) · `[1]` = conditional ← always used here
- 6,144 = 16×16×24 packed tokens

---

### Frame mean-embedding (used in PCA and similarity)

For latent frame `p`, average the 384 token embeddings → one 4096-dim vector.
Stack 16 → `[16, 4096]` → SVD → 2D scatter.

---

> ⚠️ **Memory:** Each `.pt` file is ~2.4 GB when hidden states are included.
> Samples are loaded and freed one at a time.  
> **Green solid line / diamond** = GT annotation throughout.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from trajectory_utils import *

records = load_all_records()
gt_dict = load_gt_annotations()

print(f"\nLayer indices probed: {LAYER_INDICES}")
print(f"Denoising steps saved: {STEP_INDICES}")
hs_records = [r for r in records if r["has_hs"]]
print(f"\n{len(hs_records)}/{len(records)} samples have hidden states")


---
## 3.1 — Per-Frame Activation Norm Heatmap

For each sample that has hidden states, plot a grid:
- **Rows** = transformer block L ∈ {L12, L24, L35, L47}
- **Columns** = denoising step τ ∈ {0, 8, 16, 23, 31, 39}

Each cell = mean `‖h^L(p)‖` per frame `p` (bar chart, x=frame, y=norm).

**What to look for:**
- Frames at or near the transition should have elevated norm in deep blocks (L35/L47).
- Does the norm profile sharpen around the GT frame as denoising progresses (left → right)?
- Does it sharpen with depth (top → bottom)?


In [ ]:
hs_records = [r for r in records if r["has_hs"]]
print(f"{len(hs_records)}/{len(records)} samples have hidden states.")
print("Loading one sample at a time (~2.4 GB) — may take 1–2 min total.\n")

if not hs_records:
    print("No hidden states found — skipping.")
else:
    frames = np.arange(F_PRIME)

    for r in hs_records:
        print(f"  Loading {r['sample_id']} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        gp  = gt_latent(r["sample_id"], gt_dict)
        col = r["color"]
        sid = r["sample_id"]
        print("done")

        n_layers = len(LAYER_INDICES)
        n_steps  = len(STEP_INDICES)
        fig, axes = plt.subplots(n_layers, n_steps, figsize=(n_steps*2.2, n_layers*2.0))
        gs_val = gt_dict.get(sid)
        gt_str = f"  |  GT {gs_val:.2f}s" if gs_val is not None else ""
        fig.suptitle(
            f"Level 3 — Per-frame hidden-state norm  ‖h^L(p)‖\n"
            f"{r['short_id']}{gt_str}  |  🟢 solid=GT",
            fontsize=9, fontweight="bold", color=col
        )

        all_norms = []
        for step_idx in STEP_INDICES:
            for layer_idx in LAYER_INDICES:
                h_t = hs.get(step_idx, {}).get(layer_idx)
                if h_t is not None:
                    nv = per_frame_norm(unpack_hidden(h_t.float()))
                    all_norms.append(nv)
        global_vmax = max(n.max() for n in all_norms) if all_norms else 1.0

        for si, step_idx in enumerate(STEP_INDICES):
            for li, layer_idx in enumerate(LAYER_INDICES):
                ax  = axes[li, si]
                h_t = hs.get(step_idx, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue
                nv = per_frame_norm(unpack_hidden(h_t.float()))
                ax.bar(frames, nv, width=0.75, color=col, alpha=0.8)
                if gp is not None:
                    ax.axvline(gp, color="#4CAF50", lw=1.5, ls="-", alpha=0.9)
                shade_cond(ax, F_PRIME)
                ax.set_xlim(-0.5, F_PRIME-0.5); ax.set_xticks(range(F_PRIME))
                ax.set_ylim(0, global_vmax * 1.05)
                ax.tick_params(labelsize=5)
                if li == 0: ax.set_title(f"τ={step_idx}", fontsize=7)
                if si == 0: ax.set_ylabel(f"L{layer_idx}", fontsize=7)

        plt.tight_layout()
        plt.show()

        del full_data, hs
        gc.collect()


---
## 3.2 — PCA of Transformer Frame Embeddings

**One figure per sample.** Grid: 4 rows (denoising step τ) × 4 cols (transformer block L).

Each scatter plot = 16 frame mean-embeddings projected to 2D via SVD.  
Blue=frame 0 → Red=frame 15. **★**=conditioned · **●**=free-middle · **🟢 diamond**=GT.

**Read across rows (τ: noise → clean):**
- τ=0 (noise): usually random scatter.
- τ=8 (early): coarse structure may emerge.
- τ=16 (mid): semantic content partially formed.
- τ=39 (clean): clearest — conditioned ★ frames anchor opposite ends.

**Read across columns (L12 → L47):**
- L12: mainly positional/spatial information.
- L47: semantic content dominates; expect tightest clustering.

**What to look for:**
- **Kink in the frame trajectory** near the GT frame (🟢 diamond)
- **Two clusters** (start-group vs end-group) = hard semantic separation
- **Single cluster** = block/step hasn't separated the two clip semantics yet


In [ ]:
cmap_frames = plt.get_cmap("RdYlBu_r")
hs_records  = [r for r in records if r["has_hs"]]

if not hs_records:
    print("No hidden states — skipping PCA.")
else:
    print(f"PCA: {len(hs_records)} samples × {len(PCA_TIMESTEPS)} timesteps "
          f"× {len(LAYER_INDICES)} blocks\n")

    for r in hs_records:
        sid = r["sample_id"]
        print(f"  Loading {sid} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        gp  = gt_latent(sid, gt_dict)
        col = r["color"]
        print("done")

        n_rows = len(PCA_TIMESTEPS)
        n_cols = len(LAYER_INDICES)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.3, n_rows * 3.3))

        gs_val = gt_dict.get(sid)
        gt_str = f"  |  GT={gs_val:.2f}s (p≈{gp:.1f})" if gs_val is not None else ""
        fig.suptitle(
            f"Level 3 — PCA frame embeddings  ·  {r['short_id']}\n"
            f"Rows=denoising step τ  |  Cols=transformer block L (depth)\n"
            f"{gt_str}  |  ◉=conditioned  ○=free-middle  ◆=GT",
            fontsize=9, fontweight="bold", color=col
        )

        for ri, step_idx in enumerate(PCA_TIMESTEPS):
            for ci, layer_idx in enumerate(LAYER_INDICES):
                ax  = axes[ri, ci]
                h_t = hs.get(step_idx, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue

                coords, expvar = pca_frame_embeddings(unpack_hidden(h_t.float()))

                for p in range(F_PRIME - 1):
                    ax.annotate("", xy=(coords[p+1, 0], coords[p+1, 1]),
                                xytext=(coords[p, 0], coords[p, 1]),
                                arrowprops=dict(arrowstyle="-|>", color="gray",
                                               lw=0.5, alpha=0.35))
                for p in range(F_PRIME):
                    c_val  = cmap_frames(p / (F_PRIME - 1))
                    marker = "*" if (p < K_LAT or p >= END_IDX) else "o"
                    ms     = 100 if marker == "*" else 50
                    ax.scatter(coords[p, 0], coords[p, 1], color=c_val, s=ms,
                               marker=marker, zorder=3, alpha=0.9)
                    ax.annotate(str(p), (coords[p, 0], coords[p, 1]),
                                fontsize=4.5, ha="center", va="center",
                                color="white", fontweight="bold")

                if gp is not None:
                    p_lo = int(gp); p_hi = min(p_lo + 1, F_PRIME - 1)
                    frac = gp - p_lo
                    gx   = coords[p_lo, 0] * (1 - frac) + coords[p_hi, 0] * frac
                    gy   = coords[p_lo, 1] * (1 - frac) + coords[p_hi, 1] * frac
                    ax.scatter(gx, gy, marker="D", s=80, color="#4CAF50", zorder=5,
                               edgecolors="black", linewidths=0.6)

                step_lbl = {0: "τ=0 (noise)", 8: "τ=8 (early)",
                            16: "τ=16 (mid)",  39: "τ=39 (clean)"}.get(step_idx, f"τ={step_idx}")
                ax.set_title(
                    f"L{layer_idx}  {step_lbl}\nEV:{expvar[0]:.0%}/{expvar[1]:.0%}",
                    fontsize=6.5
                )
                ax.tick_params(labelsize=5)

        plt.tight_layout()
        plt.show()

        del full_data, hs
        gc.collect()


---
## 3.3 — Frame Cosine-Similarity Matrix

For each sample, at denoising steps τ ∈ {8 (early), 16 (mid), 39 (clean)},
plot the **[16×16] frame cosine-similarity matrix** at each of the 4 transformer blocks.

Entry `(p, q)` = cos(h̄^L(p), h̄^L(q)) — similarity between the mean embeddings
of frames p and q at block L.

**Pattern guide:**

| Pattern | Meaning |
|---------|---------|
| **Block diagonal** (two green blobs + red off-diagonal) | Hard semantic boundary at the cluster edge → the dissolve frame |
| Smooth gradient (similarity decreasing with temporal distance) | Gradual transition — no hard cut |
| Uniform green | This block doesn't differentiate frames — encodes only spatial content |

**Green dashed lines** = GT annotation.
**Cyan/magenta dotted lines** = conditioning boundaries (p=3.5 and p=11.5).

**Comparing L12 → L47:** Does the block-diagonal become sharper in deeper layers
as representations become more semantically discriminative?


In [ ]:
SIM_TIMESTEPS = [8, 16, 39]
hs_records    = [r for r in records if r["has_hs"]]

if not hs_records:
    print("No hidden states — skipping similarity matrix.")
else:
    for r in hs_records:
        sid = r["sample_id"]
        print(f"Sim matrix: loading {sid} ...", end=" ", flush=True)
        full_data = torch.load(r["traj_path"], weights_only=False, map_location="cpu")
        hs  = full_data["hidden_states"]
        gp  = gt_latent(sid, gt_dict)
        col = r["color"]
        print("done")

        for target_step in SIM_TIMESTEPS:
            step_lbl = {8: "early (τ=8)", 16: "mid (τ=16)", 39: "clean (τ=39)"}[target_step]
            fig, axes = plt.subplots(1, len(LAYER_INDICES),
                                     figsize=(len(LAYER_INDICES) * 2.9, 3.0))
            gs_val = gt_dict.get(sid)
            gt_str = f"  GT {gs_val:.2f}s" if gs_val is not None else ""
            fig.suptitle(
                f"Level 3 — Frame cosine-sim  at {step_lbl}  ·  {r['short_id']}\n"
                f"🟢 dashed=GT{gt_str}  |  Block-diagonal = hard semantic cut",
                fontsize=8.5, fontweight="bold", color=col
            )

            for ci, layer_idx in enumerate(LAYER_INDICES):
                ax  = axes[ci]
                h_t = hs.get(target_step, {}).get(layer_idx)
                if h_t is None:
                    ax.set_visible(False); continue

                sim = cosine_sim_matrix(unpack_hidden(h_t.float()))
                im  = ax.imshow(sim, vmin=-0.3, vmax=1.0, cmap="RdYlGn",
                                origin="upper", interpolation="nearest")
                for bnd, c in [(K_LAT - 0.5, "#00BCD4"), (END_IDX - 0.5, "#E91E63")]:
                    ax.axhline(bnd, color=c, lw=1.0, ls=":", alpha=0.8)
                    ax.axvline(bnd, color=c, lw=1.0, ls=":", alpha=0.8)
                if gp is not None:
                    ax.axhline(gp - 0.5, color="#4CAF50", lw=1.8, ls="--", alpha=0.9)
                    ax.axvline(gp - 0.5, color="#4CAF50", lw=1.8, ls="--", alpha=0.9)
                ax.set_xticks(range(0, F_PRIME, 4))
                ax.set_yticks(range(0, F_PRIME, 4))
                ax.tick_params(labelsize=6.5)
                ax.set_title(f"L{layer_idx}", fontsize=9)
                ax.set_xlabel("frame p", fontsize=7)
                if ci == 0:
                    ax.set_ylabel("frame q", fontsize=7)
                plt.colorbar(im, ax=ax, shrink=0.9, pad=0.02)

            plt.tight_layout()
            plt.show()

        del full_data, hs
        gc.collect()
